# Using StatsBomb Open Data

This notebook demonstrates how to load, explore, and visualise football event data from [StatsBomb Open Data](https://github.com/statsbomb/open-data).

## Setup

Install the required packages:
```bash
pip install statsbombpy matplotlib pandas
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

%matplotlib inline
plt.style.use('ggplot')

## 1. Loading StatsBomb Data

We use the `statsbombpy` library to access free competition and match data.

In [ ]:
from statsbombpy import sb

# List available competitions
competitions = sb.competitions()
competitions.head(10)

In [ ]:
# Select a competition and season (e.g., FIFA World Cup 2022)
comp_id = 43   # FIFA World Cup
season_id = 106  # 2022

matches = sb.matches(competition_id=comp_id, season_id=season_id)
print(f'Number of matches: {len(matches)}')
matches[['match_id', 'home_team', 'away_team', 'home_score', 'away_score']].head(10)

## 2. Match Events

Load detailed event data for a specific match.

In [ ]:
# Pick the first match
match_id = matches.iloc[0]['match_id']
events = sb.events(match_id=match_id)

print(f'Total events: {len(events)}')
print(f'Event types: {events["type"].nunique()}')
events['type'].value_counts().head(10)

## 3. Drawing a Football Pitch

A helper function to draw a standard pitch (StatsBomb coordinates: 120 × 80).

In [ ]:
def draw_pitch(ax=None, colour='black', linewidth=1.5):
    """Draw a football pitch using StatsBomb dimensions (120 x 80)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 8))

    # Pitch outline
    ax.plot([0, 120], [0, 0], color=colour, linewidth=linewidth)
    ax.plot([0, 120], [80, 80], color=colour, linewidth=linewidth)
    ax.plot([0, 0], [0, 80], color=colour, linewidth=linewidth)
    ax.plot([120, 120], [0, 80], color=colour, linewidth=linewidth)

    # Halfway line and centre circle
    ax.plot([60, 60], [0, 80], color=colour, linewidth=linewidth)
    centre_circle = plt.Circle((60, 40), 10, fill=False, color=colour, linewidth=linewidth)
    ax.add_patch(centre_circle)
    ax.plot(60, 40, 'o', color=colour, markersize=4)

    # Left penalty area
    ax.plot([0, 18], [18, 18], color=colour, linewidth=linewidth)
    ax.plot([0, 18], [62, 62], color=colour, linewidth=linewidth)
    ax.plot([18, 18], [18, 62], color=colour, linewidth=linewidth)

    # Right penalty area
    ax.plot([102, 120], [18, 18], color=colour, linewidth=linewidth)
    ax.plot([102, 120], [62, 62], color=colour, linewidth=linewidth)
    ax.plot([102, 102], [18, 62], color=colour, linewidth=linewidth)

    # Left 6-yard box
    ax.plot([0, 6], [30, 30], color=colour, linewidth=linewidth)
    ax.plot([0, 6], [50, 50], color=colour, linewidth=linewidth)
    ax.plot([6, 6], [30, 50], color=colour, linewidth=linewidth)

    # Right 6-yard box
    ax.plot([114, 120], [30, 30], color=colour, linewidth=linewidth)
    ax.plot([114, 120], [50, 50], color=colour, linewidth=linewidth)
    ax.plot([114, 114], [30, 50], color=colour, linewidth=linewidth)

    # Penalty spots
    ax.plot(12, 40, 'o', color=colour, markersize=4)
    ax.plot(108, 40, 'o', color=colour, markersize=4)

    # Penalty arcs
    left_arc = Arc((12, 40), 18.3, 18.3, angle=0, theta1=310, theta2=50, color=colour, linewidth=linewidth)
    right_arc = Arc((108, 40), 18.3, 18.3, angle=0, theta1=130, theta2=230, color=colour, linewidth=linewidth)
    ax.add_patch(left_arc)
    ax.add_patch(right_arc)

    ax.set_xlim(-2, 122)
    ax.set_ylim(-2, 82)
    ax.set_aspect('equal')
    ax.axis('off')
    return ax

## 4. Shot Map

In [ ]:
shots = events[events['type'] == 'Shot'].copy()

if not shots.empty and 'location' in shots.columns:
    shots['x'] = shots['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else None)
    shots['y'] = shots['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else None)
    shots = shots.dropna(subset=['x', 'y'])

    fig, ax = plt.subplots(figsize=(12, 8))
    draw_pitch(ax=ax)

    for _, shot in shots.iterrows():
        colour = '#2ecc71' if shot.get('shot_outcome', '') == 'Goal' else '#e74c3c'
        size = shot.get('shot_statsbomb_xg', 0.1) * 500 + 30
        ax.scatter(shot['x'], shot['y'], s=size, c=colour, alpha=0.7, edgecolors='black', zorder=5)

    ax.set_title('Shot Map (green = goal, red = no goal; size ∝ xG)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No shot data available for this match.')

## 5. Pass Network

In [ ]:
passes = events[events['type'] == 'Pass'].copy()

if not passes.empty and 'location' in passes.columns:
    # Filter to one team
    team_name = passes['team'].value_counts().index[0]
    team_passes = passes[passes['team'] == team_name].copy()

    team_passes['x'] = team_passes['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else None)
    team_passes['y'] = team_passes['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else None)
    team_passes = team_passes.dropna(subset=['x', 'y'])

    # Average positions
    avg_pos = team_passes.groupby('player')[['x', 'y']].mean()

    fig, ax = plt.subplots(figsize=(12, 8))
    draw_pitch(ax=ax)

    ax.scatter(avg_pos['x'], avg_pos['y'], s=200, c='#3498db', edgecolors='black', zorder=5)
    for player, pos in avg_pos.iterrows():
        ax.annotate(player.split()[-1], (pos['x'], pos['y']),
                    textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=8, fontweight='bold')

    ax.set_title(f'Average Positions – {team_name}', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No pass data available for this match.')

## 6. Summary Statistics

In [ ]:
summary = events.groupby(['team', 'type']).size().unstack(fill_value=0)

key_actions = ['Pass', 'Shot', 'Dribble', 'Foul Committed', 'Ball Recovery']
available = [a for a in key_actions if a in summary.columns]

if available:
    print('Key Action Counts by Team')
    print('=' * 50)
    print(summary[available])
else:
    print('No matching action types found.')

## Next Steps

- Extend the analysis to multiple matches or a full tournament.
- Build expected goals (xG) models using shot features.
- Create pass networks and analyse passing patterns.